Cell 1 — Setup konfigurasi path dan dataset

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# =========================
# CONFIG: INPUT CLEAN DATASET
# =========================

CLEAN_DEV_DIR = Path("/media/spell/Spell-lab/Lidar/B.Clean Dataset")
CLEAN_TEST_DIR = Path("/media/spell/Spell-lab/Lidar/B.Clean Testing")

# =========================
# CONFIG: OUTPUT FRAMED DATASET
# =========================

FRAMED_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/C.Framed Dataset")

FRAMED_DEV_DIR = FRAMED_BASE_DIR / "Dataset Development"
FRAMED_TEST_DIR = FRAMED_BASE_DIR / "Dataset Testing"

REPORT_DIR = FRAMED_BASE_DIR / "_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# DATASET STRUCTURE
# =========================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia",
    "Afi",
    "Aswangga",
    "Bustan",
    "Dilia",
    "Eldivo",
    "Fathir",
    "Lina",
    "Manda",
    "Miftah",
    "Teguh",
    "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya",
    "Naila",
    "Nana",
    "Rega",
    "Zaira",
]

FILES = [f"{i}.csv" for i in range(1, 10)]

REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity"]
OUTPUT_COLUMNS = ["frame_id", "Timestamp", "X", "Y", "Z", "Reflectivity"]

# =========================
# FRAME CONSTRUCTION CONFIG
# =========================

TARGET_FRAME_MS = 100.0
EXPECTED_POINTS_PER_PACKET = 96

expected_dev_files = len(ACTIVITIES) * len(DEV_SUBJECTS) * len(FILES)
expected_test_files = len(TEST_ROOMS) * len(ACTIVITIES) * len(TEST_SUBJECTS) * len(FILES)
expected_total_files = expected_dev_files + expected_test_files

print("===== FRAME BUILDER CONFIG =====")
print("Clean development dir:", CLEAN_DEV_DIR)
print("Clean testing dir    :", CLEAN_TEST_DIR)
print("Framed output dir    :", FRAMED_BASE_DIR)
print("Report dir           :", REPORT_DIR)

print("\nExpected development files:", expected_dev_files)
print("Expected testing files    :", expected_test_files)
print("Expected total files      :", expected_total_files)

print("\nRequired columns:", REQUIRED_COLUMNS)
print("Output columns  :", OUTPUT_COLUMNS)
print("Target frame ms :", TARGET_FRAME_MS)
print("Expected point per packet:", EXPECTED_POINTS_PER_PACKET)

print("\nPath exists check:")
print("Clean development exists:", CLEAN_DEV_DIR.exists())
print("Clean testing exists    :", CLEAN_TEST_DIR.exists())

===== FRAME BUILDER CONFIG =====
Clean development dir: /media/spell/Spell-lab/Lidar/B.Clean Dataset
Clean testing dir    : /media/spell/Spell-lab/Lidar/B.Clean Testing
Framed output dir    : /media/spell/Spell-lab/Lidar/C.Framed Dataset
Report dir           : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports

Expected development files: 432
Expected testing files    : 360
Expected total files      : 792

Required columns: ['Timestamp', 'X', 'Y', 'Z', 'Reflectivity']
Output columns  : ['frame_id', 'Timestamp', 'X', 'Y', 'Z', 'Reflectivity']
Target frame ms : 100.0
Expected point per packet: 96

Path exists check:
Clean development exists: True
Clean testing exists    : True


Cell 2 — Function frame builder untuk satu file

In [8]:
def build_frame_for_one_file(
    input_path: Path,
    output_path: Path,
    dataset_type: str,
    activity: str,
    subject: str,
    file_name: str,
    room: str = None,
    target_frame_ms: float = 100.0,
    expected_points_per_packet: int = 96,
):
    """
    Build analysis frames from Livox packet-level timestamps.
    
    Input columns:
        Timestamp, X, Y, Z, Reflectivity
    
    Output columns:
        frame_id, Timestamp, X, Y, Z, Reflectivity
    
    Core logic:
        1 unique Timestamp = 1 packet
        1 packet = 96 points
        packet interval estimated from median Timestamp diff
        packet_per_frame = round(target_frame_ms / packet_interval_ms)
        partial frames are discarded
        frame_id is reindexed from 0
    """
    
    result = {
        "dataset_type": dataset_type,
        "room": room,
        "activity": activity,
        "subject": subject,
        "file_name": file_name,
        "input_path": str(input_path),
        "output_path": str(output_path),
        "input_exists": input_path.exists(),
        "success": False,
        "error": None,
        
        "rows_input": None,
        "rows_after_numeric_clean": None,
        "rows_output": None,
        
        "missing_columns": None,
        "num_unique_packets": None,
        "points_per_packet_min": None,
        "points_per_packet_median": None,
        "points_per_packet_max": None,
        "points_per_packet_is_expected": None,
        
        "timestamp_diff_unique_count": None,
        "timestamp_diff_mode_ns": None,
        "timestamp_diff_median_ns": None,
        "abnormal_gap_count": None,
        
        "packet_per_frame": None,
        "nominal_frame_duration_ms": None,
        "nominal_analysis_fps": None,
        
        "num_frames_before_drop": None,
        "num_partial_frames": None,
        "partial_frame_ids": None,
        "partial_only_at_end": None,
        "num_frames_after_drop": None,
        
        "final_point_count_unique": None,
        "final_point_count_min": None,
        "final_point_count_median": None,
        "final_point_count_max": None,
    }
    
    try:
        if not input_path.exists():
            result["error"] = "Input file not found"
            return result
        
        df = pd.read_csv(input_path)
        result["rows_input"] = len(df)
        
        # =========================
        # Validate required columns
        # =========================
        missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
        result["missing_columns"] = missing_cols
        
        if missing_cols:
            result["error"] = f"Missing required columns: {missing_cols}"
            return result
        
        # =========================
        # Keep only required columns, do numeric safety check
        # Tidak mengubah makna kolom inti, hanya memastikan numeric valid
        # =========================
        df = df[REQUIRED_COLUMNS].copy()
        
        for col in REQUIRED_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        
        df = df.dropna(subset=REQUIRED_COLUMNS).copy()
        df = df.sort_values("Timestamp").reset_index(drop=True)
        
        result["rows_after_numeric_clean"] = len(df)
        
        if len(df) == 0:
            result["error"] = "No valid rows after numeric cleaning"
            return result
        
        # =========================
        # Unique timestamp = packet
        # =========================
        unique_timestamps = np.sort(df["Timestamp"].unique())
        result["num_unique_packets"] = int(len(unique_timestamps))
        
        if len(unique_timestamps) < 2:
            result["error"] = "Not enough unique timestamps to build frames"
            return result
        
        # =========================
        # Points per packet audit
        # =========================
        points_per_packet = df.groupby("Timestamp").size()
        
        ppp_min = int(points_per_packet.min())
        ppp_median = float(points_per_packet.median())
        ppp_max = int(points_per_packet.max())
        
        result["points_per_packet_min"] = ppp_min
        result["points_per_packet_median"] = ppp_median
        result["points_per_packet_max"] = ppp_max
        result["points_per_packet_is_expected"] = bool(
            ppp_min == expected_points_per_packet and ppp_max == expected_points_per_packet
        )
        
        # =========================
        # Timestamp diff audit
        # =========================
        timestamp_diff = np.diff(unique_timestamps)
        
        timestamp_diff_series = pd.Series(timestamp_diff)
        diff_counts = timestamp_diff_series.value_counts().sort_values(ascending=False)
        
        median_dt_ns = float(np.median(timestamp_diff))
        mode_dt_ns = float(diff_counts.index[0])
        
        result["timestamp_diff_unique_count"] = int(timestamp_diff_series.nunique())
        result["timestamp_diff_mode_ns"] = mode_dt_ns
        result["timestamp_diff_median_ns"] = median_dt_ns
        
        # Abnormal gap pakai median sebagai expected packet interval
        # Gunakan isclose agar aman kalau Timestamp terbaca float.
        abnormal_gap = ~np.isclose(timestamp_diff, median_dt_ns, rtol=0, atol=1e-6)
        abnormal_gap_count = int(abnormal_gap.sum())
        result["abnormal_gap_count"] = abnormal_gap_count
        
        # =========================
        # Packet per frame
        # =========================
        median_dt_sec = median_dt_ns / 1e9
        target_frame_sec = target_frame_ms / 1000.0
        
        if median_dt_sec <= 0:
            result["error"] = "Invalid median timestamp interval"
            return result
        
        packet_per_frame = int(round(target_frame_sec / median_dt_sec))
        
        if packet_per_frame <= 0:
            result["error"] = "Invalid packet_per_frame"
            return result
        
        nominal_frame_duration_sec = packet_per_frame * median_dt_sec
        nominal_frame_duration_ms = nominal_frame_duration_sec * 1000.0
        nominal_analysis_fps = 1.0 / nominal_frame_duration_sec
        
        result["packet_per_frame"] = int(packet_per_frame)
        result["nominal_frame_duration_ms"] = float(nominal_frame_duration_ms)
        result["nominal_analysis_fps"] = float(nominal_analysis_fps)
        
        # =========================
        # Assign packet index and frame_id
        # =========================
        packet_df = pd.DataFrame({"Timestamp": unique_timestamps})
        packet_df["packet_idx"] = np.arange(len(packet_df))
        packet_df["frame_id"] = packet_df["packet_idx"] // packet_per_frame
        
        df_framed = df.merge(
            packet_df[["Timestamp", "packet_idx", "frame_id"]],
            on="Timestamp",
            how="left"
        )
        
        if df_framed["frame_id"].isna().any():
            result["error"] = "frame_id merge produced NaN"
            return result
        
        # =========================
        # Frame time and partial frame audit
        # =========================
        frame_time_info = packet_df.groupby("frame_id").agg(
            start_timestamp=("Timestamp", "min"),
            end_timestamp=("Timestamp", "max"),
            packet_count=("Timestamp", "count")
        ).reset_index()
        
        frame_time_info["observed_span_ns"] = frame_time_info["end_timestamp"] - frame_time_info["start_timestamp"]
        frame_time_info["observed_span_ms"] = frame_time_info["observed_span_ns"] / 1e6
        frame_time_info["nominal_duration_ms"] = frame_time_info["packet_count"] * (median_dt_ns / 1e6)
        frame_time_info["is_partial"] = frame_time_info["packet_count"] < packet_per_frame
        
        result["num_frames_before_drop"] = int(frame_time_info["frame_id"].nunique())
        result["num_partial_frames"] = int(frame_time_info["is_partial"].sum())
        
        partial_ids = frame_time_info.loc[frame_time_info["is_partial"], "frame_id"].astype(int).tolist()
        last_frame_id = int(frame_time_info["frame_id"].max())
        
        result["partial_frame_ids"] = json.dumps(partial_ids)
        result["partial_only_at_end"] = bool((len(partial_ids) == 0) or (partial_ids == [last_frame_id]))
        
        # =========================
        # Drop partial frames
        # =========================
        valid_frame_ids = frame_time_info.loc[
            frame_time_info["is_partial"] == False,
            "frame_id"
        ]
        
        df_framed_valid = df_framed[df_framed["frame_id"].isin(valid_frame_ids)].copy()
        
        if len(df_framed_valid) == 0:
            result["error"] = "No valid full frames after dropping partial frames"
            return result
        
        # =========================
        # Reindex frame_id from 0
        # =========================
        old_to_new_frame = {
            old_id: new_id
            for new_id, old_id in enumerate(sorted(df_framed_valid["frame_id"].unique()))
        }
        
        df_framed_valid["frame_id"] = df_framed_valid["frame_id"].map(old_to_new_frame).astype(int)
        
        result["num_frames_after_drop"] = int(df_framed_valid["frame_id"].nunique())
        
        # =========================
        # Final output columns
        # =========================
        df_final = df_framed_valid[OUTPUT_COLUMNS].copy()
        
        # Safety: ensure original feature values not modified except frame_id addition and row sorting/filtering.
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df_final.to_csv(
            output_path,
            index=False,
            encoding="utf-8",
            sep=",",
            lineterminator="\n"
        )
        
        result["rows_output"] = int(len(df_final))
        
        # =========================
        # Final point count audit
        # =========================
        final_points_per_frame = df_final.groupby("frame_id").size()
        final_point_counts = final_points_per_frame.value_counts().sort_index()
        
        result["final_point_count_unique"] = json.dumps({
            int(k): int(v) for k, v in final_point_counts.items()
        })
        result["final_point_count_min"] = int(final_points_per_frame.min())
        result["final_point_count_median"] = float(final_points_per_frame.median())
        result["final_point_count_max"] = int(final_points_per_frame.max())
        
        result["success"] = True
        return result
    
    except Exception as e:
        result["error"] = str(e)
        return result


print("Function build_frame_for_one_file() sudah siap.")
print("Fungsi ini tidak overwrite Clean Dataset, hanya membuat output baru ke Framed Dataset.")

Function build_frame_for_one_file() sudah siap.
Fungsi ini tidak overwrite Clean Dataset, hanya membuat output baru ke Framed Dataset.


Cell 3 — Build daftar semua file development dan testing

In [9]:
# =========================
# BUILD FILE JOB LIST
# =========================

jobs = []

# Development dataset
for activity in ACTIVITIES:
    for subject in DEV_SUBJECTS:
        for file_name in FILES:
            input_path = CLEAN_DEV_DIR / activity / subject / file_name
            output_path = FRAMED_DEV_DIR / activity / subject / file_name
            
            jobs.append({
                "dataset_type": "development",
                "room": None,
                "activity": activity,
                "subject": subject,
                "file_name": file_name,
                "input_path": input_path,
                "output_path": output_path,
            })

# Testing dataset
for room in TEST_ROOMS:
    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            for file_name in FILES:
                input_path = CLEAN_TEST_DIR / room / activity / subject / file_name
                output_path = FRAMED_TEST_DIR / room / activity / subject / file_name
                
                jobs.append({
                    "dataset_type": "testing",
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "file_name": file_name,
                    "input_path": input_path,
                    "output_path": output_path,
                })

jobs_df = pd.DataFrame([
    {
        "dataset_type": j["dataset_type"],
        "room": j["room"],
        "activity": j["activity"],
        "subject": j["subject"],
        "file_name": j["file_name"],
        "input_path": str(j["input_path"]),
        "output_path": str(j["output_path"]),
        "input_exists": j["input_path"].exists(),
    }
    for j in jobs
])

print("===== FRAME BUILDER JOB LIST =====")
print("Total jobs:", len(jobs_df))
print("Expected total files:", expected_total_files)
print("Input exists:", int(jobs_df["input_exists"].sum()))
print("Input missing:", int((~jobs_df["input_exists"]).sum()))

print("\nSummary by dataset type:")
display(
    jobs_df.groupby("dataset_type")["input_exists"]
    .agg(expected="count", found="sum")
    .assign(missing=lambda x: x["expected"] - x["found"])
    .reset_index()
)

print("\nMissing input preview:")
display(jobs_df[jobs_df["input_exists"] == False].head(30))

===== FRAME BUILDER JOB LIST =====
Total jobs: 792
Expected total files: 792
Input exists: 792
Input missing: 0

Summary by dataset type:


,dataset_type,expected,found,missing
0,development,432,432,0
1,testing,360,360,0



Missing input preview:


,dataset_type,room,activity,subject,file_name,input_path,output_path,input_exists


Cell 4 — Jalankan batch frame construction ke seluruh dataset

In [10]:
# =========================
# RUN BATCH FRAME CONSTRUCTION
# =========================

results = []

for idx, job in enumerate(jobs, start=1):
    res = build_frame_for_one_file(
        input_path=job["input_path"],
        output_path=job["output_path"],
        dataset_type=job["dataset_type"],
        room=job["room"],
        activity=job["activity"],
        subject=job["subject"],
        file_name=job["file_name"],
        target_frame_ms=TARGET_FRAME_MS,
        expected_points_per_packet=EXPECTED_POINTS_PER_PACKET,
    )
    
    results.append(res)
    
    if idx % 50 == 0 or idx == len(jobs):
        print(f"Processed {idx}/{len(jobs)} files...")

frame_report_df = pd.DataFrame(results)

success_count = int(frame_report_df["success"].sum())
failed_count = int((~frame_report_df["success"]).sum())

print("===== BATCH FRAME CONSTRUCTION SUMMARY =====")
print("Total files processed:", len(frame_report_df))
print("Success files        :", success_count)
print("Failed files         :", failed_count)

print("\nSuccess by dataset type:")
display(
    frame_report_df.groupby("dataset_type")["success"]
    .agg(total="count", success="sum")
    .assign(failed=lambda x: x["total"] - x["success"])
    .reset_index()
)

print("\nFailed files preview:")
display(frame_report_df[frame_report_df["success"] == False].head(30))

Processed 50/792 files...
Processed 100/792 files...
Processed 150/792 files...
Processed 200/792 files...
Processed 250/792 files...
Processed 300/792 files...
Processed 350/792 files...
Processed 400/792 files...
Processed 450/792 files...
Processed 500/792 files...
Processed 550/792 files...
Processed 600/792 files...
Processed 650/792 files...
Processed 700/792 files...
Processed 750/792 files...
Processed 792/792 files...
===== BATCH FRAME CONSTRUCTION SUMMARY =====
Total files processed: 792
Success files        : 792
Failed files         : 0

Success by dataset type:


,dataset_type,total,success,failed
0,development,432,432,0
1,testing,360,360,0



Failed files preview:


,dataset_type,room,activity,subject,file_name,input_path,output_path,input_exists,success,error,...,nominal_analysis_fps,num_frames_before_drop,num_partial_frames,partial_frame_ids,partial_only_at_end,num_frames_after_drop,final_point_count_unique,final_point_count_min,final_point_count_median,final_point_count_max


Cell 5 — Audit hasil frame construction

In [11]:
# =========================
# AUDIT FRAME CONSTRUCTION RESULTS
# =========================

successful_df = frame_report_df[frame_report_df["success"] == True].copy()

print("===== FRAME AUDIT OVERVIEW =====")
print("Successful files:", len(successful_df))
print("Failed files    :", int((~frame_report_df["success"]).sum()))

if len(successful_df) > 0:
    print("\nPacket per frame distribution:")
    display(successful_df["packet_per_frame"].value_counts().sort_index())

    print("\nNominal frame duration ms:")
    display(successful_df["nominal_frame_duration_ms"].describe())

    print("\nNominal analysis FPS:")
    display(successful_df["nominal_analysis_fps"].describe())

    print("\nPoints per packet median:")
    display(successful_df["points_per_packet_median"].describe())

    print("\nTimestamp diff median ns:")
    display(successful_df["timestamp_diff_median_ns"].describe())

    print("\nAbnormal timestamp gap count distribution:")
    display(successful_df["abnormal_gap_count"].value_counts().sort_index())

    print("\nPartial frame count distribution:")
    display(successful_df["num_partial_frames"].value_counts().sort_index())

    print("\nPartial only at end distribution:")
    display(successful_df["partial_only_at_end"].value_counts())

    print("\nFinal point count median per frame:")
    display(successful_df["final_point_count_median"].describe())

# Problem audit: success tapi ada warning penting
warning_df = successful_df[
    (successful_df["points_per_packet_is_expected"] == False) |
    (successful_df["abnormal_gap_count"] > 0) |
    (successful_df["partial_only_at_end"] == False) |
    (successful_df["packet_per_frame"] != 208)
].copy()

print("\n===== WARNING FILES =====")
print("Warning files:", len(warning_df))
display(warning_df.head(30))

===== FRAME AUDIT OVERVIEW =====
Successful files: 792
Failed files    : 0

Packet per frame distribution:


packet_per_frame
208    792
Name: count, dtype: int64


Nominal frame duration ms:


count    792.00
mean      99.84
std        0.00
min       99.84
25%       99.84
50%       99.84
75%       99.84
max       99.84
Name: nominal_frame_duration_ms, dtype: float64


Nominal analysis FPS:


...

partial_only_at_end
True    792
Name: count, dtype: int64


Final point count median per frame:


count      792.0
mean     19968.0
std          0.0
min      19968.0
25%      19968.0
50%      19968.0
75%      19968.0
max      19968.0
Name: final_point_count_median, dtype: float64


===== WARNING FILES =====
Warning files: 382


,dataset_type,room,activity,subject,file_name,input_path,output_path,input_exists,success,error,...,nominal_analysis_fps,num_frames_before_drop,num_partial_frames,partial_frame_ids,partial_only_at_end,num_frames_after_drop,final_point_count_unique,final_point_count_min,final_point_count_median,final_point_count_max
0,development,None,Bungkuk,Adelia,1.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,56,1,[55],True,55,"{""19968"": 55}",19968,19968.0,19968
1,development,None,Bungkuk,Adelia,2.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,55,1,[54],True,54,"{""19968"": 54}",19968,19968.0,19968
2,development,None,Bungkuk,Adelia,3.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,57,1,[56],True,56,"{""19968"": 56}",19968,19968.0,19968
3,development,None,Bungkuk,Adelia,4.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,59,1,[58],True,58,"{""19968"": 58}",19968,19968.0,19968
4,development,None,Bungkuk,Adelia,5.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,57,1,[56],True,56,"{""19968"": 56}",19968,19968.0,19968
5,development,None,Bungkuk,Adelia,6.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,62,1,[61],True,61,"{""19968"": 61}",19968,19968.0,19968
9,development,None,Bungkuk,Afi,1.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,56,1,[55],True,55,"{""19968"": 55}",19968,19968.0,19968
10,development,None,Bungkuk,Afi,2.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,55,1,[54],True,54,"{""19968"": 54}",19968,19968.0,19968
18,development,None,Bungkuk,Aswangga,1.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,56,1,[55],True,55,"{""19968"": 55}",19968,19968.0,19968
19,development,None,Bungkuk,Aswangga,2.csv,/media/spell/Spell-lab/Lidar/B.Clean Dataset/B...,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...,True,True,None,...,10.016026,61,1,[60],True,60,"{""19968"": 60}",19968,19968.0,19968


Cell 6 — Validasi output file framed bisa dibaca dan kolomnya benar

In [12]:
# =========================
# VALIDATE OUTPUT FRAMED FILES
# =========================

validation_records = []

for _, row in frame_report_df.iterrows():
    output_path = Path(row["output_path"])
    
    rec = {
        "dataset_type": row["dataset_type"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_name": row["file_name"],
        "output_path": row["output_path"],
        "exists": output_path.exists(),
        "can_read": False,
        "columns_exact_match": False,
        "num_rows": None,
        "num_frames": None,
        "frame_id_min": None,
        "frame_id_max": None,
        "has_na": None,
        "error": None,
    }
    
    try:
        if not output_path.exists():
            rec["error"] = "Output file not found"
            validation_records.append(rec)
            continue
        
        df_check = pd.read_csv(output_path)
        
        rec["can_read"] = True
        rec["columns_exact_match"] = list(df_check.columns) == OUTPUT_COLUMNS
        rec["num_rows"] = int(len(df_check))
        rec["num_frames"] = int(df_check["frame_id"].nunique()) if "frame_id" in df_check.columns else None
        rec["frame_id_min"] = int(df_check["frame_id"].min()) if "frame_id" in df_check.columns and len(df_check) > 0 else None
        rec["frame_id_max"] = int(df_check["frame_id"].max()) if "frame_id" in df_check.columns and len(df_check) > 0 else None
        rec["has_na"] = bool(df_check[OUTPUT_COLUMNS].isna().any().any()) if rec["columns_exact_match"] else None
        
    except Exception as e:
        rec["error"] = str(e)
    
    validation_records.append(rec)

framed_validation_df = pd.DataFrame(validation_records)

problem_validation_df = framed_validation_df[
    (framed_validation_df["exists"] == False) |
    (framed_validation_df["can_read"] == False) |
    (framed_validation_df["columns_exact_match"] == False) |
    (framed_validation_df["has_na"] == True)
].copy()

print("===== FRAMED OUTPUT VALIDATION SUMMARY =====")
print("Expected files      :", len(framed_validation_df))
print("Existing files      :", int(framed_validation_df["exists"].sum()))
print("Readable files      :", int(framed_validation_df["can_read"].sum()))
print("Exact column match  :", int(framed_validation_df["columns_exact_match"].sum()))
print("Files with NA       :", int((framed_validation_df["has_na"] == True).sum()))
print("Problem files       :", len(problem_validation_df))

print("\nProblem validation preview:")
display(problem_validation_df.head(30))

print("\nFrame count statistics:")
display(framed_validation_df["num_frames"].describe())

===== FRAMED OUTPUT VALIDATION SUMMARY =====
Expected files      : 792
Existing files      : 792
Readable files      : 792
Exact column match  : 792
Files with NA       : 0
Problem files       : 0

Problem validation preview:


,dataset_type,room,activity,subject,file_name,output_path,exists,can_read,columns_exact_match,num_rows,num_frames,frame_id_min,frame_id_max,has_na,error



Frame count statistics:


count    792.000000
mean      59.811869
std        7.443591
min       35.000000
25%       56.000000
50%       58.000000
75%       64.000000
max      113.000000
Name: num_frames, dtype: float64

Cell 7 — Simpan semua report

In [13]:
# =========================
# SAVE REPORTS
# =========================

frame_report_path = REPORT_DIR / "frame_builder_report_detail.csv"
warning_report_path = REPORT_DIR / "frame_builder_warning_files.csv"
validation_report_path = REPORT_DIR / "frame_builder_output_validation.csv"
validation_problem_path = REPORT_DIR / "frame_builder_output_problem_files.csv"

summary_by_dataset_path = REPORT_DIR / "frame_builder_summary_by_dataset.csv"
summary_by_activity_path = REPORT_DIR / "frame_builder_summary_by_dataset_activity.csv"

frame_report_df.to_csv(frame_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
warning_df.to_csv(warning_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
framed_validation_df.to_csv(validation_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
problem_validation_df.to_csv(validation_problem_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")

summary_by_dataset = (
    frame_report_df
    .groupby("dataset_type")
    .agg(
        total_files=("file_name", "count"),
        success_files=("success", "sum"),
        failed_files=("success", lambda x: int((~x).sum())),
        mean_frames_after_drop=("num_frames_after_drop", "mean"),
        median_frames_after_drop=("num_frames_after_drop", "median"),
        mean_nominal_fps=("nominal_analysis_fps", "mean"),
        warning_abnormal_gap_files=("abnormal_gap_count", lambda x: int((x > 0).sum())),
        warning_partial_not_end_files=("partial_only_at_end", lambda x: int((x == False).sum())),
    )
    .reset_index()
)

summary_by_activity = (
    frame_report_df
    .groupby(["dataset_type", "activity"])
    .agg(
        total_files=("file_name", "count"),
        success_files=("success", "sum"),
        failed_files=("success", lambda x: int((~x).sum())),
        mean_frames_after_drop=("num_frames_after_drop", "mean"),
        median_frames_after_drop=("num_frames_after_drop", "median"),
    )
    .reset_index()
)

summary_by_dataset.to_csv(summary_by_dataset_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_by_activity.to_csv(summary_by_activity_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")

print("===== REPORTS SAVED =====")
print("Detail report             :", frame_report_path)
print("Warning report            :", warning_report_path)
print("Validation report         :", validation_report_path)
print("Validation problem report :", validation_problem_path)
print("Summary by dataset        :", summary_by_dataset_path)
print("Summary by activity       :", summary_by_activity_path)

print("\n===== SUMMARY BY DATASET =====")
display(summary_by_dataset)

print("\n===== SUMMARY BY DATASET & ACTIVITY =====")
display(summary_by_activity)

print("\nFinal status:")
print("Total expected files:", expected_total_files)
print("Frame success files :", int(frame_report_df["success"].sum()))
print("Frame failed files  :", int((~frame_report_df["success"]).sum()))
print("Warning files       :", len(warning_df))
print("Validation problems :", len(problem_validation_df))

===== REPORTS SAVED =====
Detail report             : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_report_detail.csv
Warning report            : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_warning_files.csv
Validation report         : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_output_validation.csv
Validation problem report : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_output_problem_files.csv
Summary by dataset        : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_summary_by_dataset.csv
Summary by activity       : /media/spell/Spell-lab/Lidar/C.Framed Dataset/_reports/frame_builder_summary_by_dataset_activity.csv

===== SUMMARY BY DATASET =====


,dataset_type,total_files,success_files,failed_files,mean_frames_after_drop,median_frames_after_drop,mean_nominal_fps,warning_abnormal_gap_files,warning_partial_not_end_files
0,development,432,432,0,61.319444,60.0,10.016026,171,0
1,testing,360,360,0,58.002778,57.0,10.016026,211,0



===== SUMMARY BY DATASET & ACTIVITY =====


,dataset_type,activity,total_files,success_files,failed_files,mean_frames_after_drop,median_frames_after_drop
0,development,Bungkuk,108,108,0,60.250000,58.0
1,development,Duduk,108,108,0,59.342593,58.0
2,development,Jatuh,108,108,0,64.157407,63.0
3,development,Jongkok,108,108,0,61.527778,60.0
4,testing,Bungkuk,90,90,0,56.433333,57.0
5,testing,Duduk,90,90,0,57.288889,57.0
6,testing,Jatuh,90,90,0,61.433333,60.0
7,testing,Jongkok,90,90,0,56.855556,57.0



Final status:
Total expected files: 792
Frame success files : 792
Frame failed files  : 0
Warning files       : 382
Validation problems : 0


Target ideal setelah semua cell dijalankan:

Frame success files : 792

Frame failed files  : 0

Warning files       : 0

Validation problems : 0

In [14]:
failed_df = frame_report_df[frame_report_df["success"] == False].copy()

display(
    failed_df[
        [
            "dataset_type", "room", "activity", "subject", "file_name",
            "input_path", "rows_input", "rows_after_numeric_clean",
            "num_unique_packets", "error"
        ]
    ]
)

print("Failed by activity:")
display(failed_df.groupby(["dataset_type", "activity", "subject"]).size().reset_index(name="failed_files"))

,dataset_type,room,activity,subject,file_name,input_path,rows_input,rows_after_numeric_clean,num_unique_packets,error


Failed by activity:


,dataset_type,activity,subject,failed_files
